# Final model, test evaluation, and API preparation

Refit the validation-selected model on the combined training and validation splits, evaluate it once on the group-disjoint supplied test set, package the pipeline, and verify the API contract.

In [1]:
from ast import literal_eval
from pathlib import Path
import sys

import joblib
import pandas as pd
from fastapi.testclient import TestClient
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.train import (
    build_logistic_regression_pipeline,
    build_svm_pipeline,
)

C:\Users\jadka\OneDrive\Documents\progressSoft_internship\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [2]:
processed_dir = PROJECT_ROOT / "data/processed"
train_df = pd.read_csv(processed_dir / "train_clean.csv")
validation_df = pd.read_csv(processed_dir / "validation_clean.csv")
test_df = pd.read_csv(processed_dir / "test_clean.csv")

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

Training shape: (50301, 4)
Validation shape: (12682, 4)
Test shape: (998, 4)


In [3]:
results_dir = PROJECT_ROOT / "reports/results"
comparison = pd.read_csv(results_dir / "tuned_model_comparison.csv")
tuned_models = comparison[comparison["model"].str.startswith("Tuned")]
selected_model_name = tuned_models.sort_values(
    "macro_f1", ascending=False
).iloc[0]["model"]

best_parameters = pd.read_csv(results_dir / "best_hyperparameters.csv")
parameter_row = best_parameters.loc[
    best_parameters["model"].map(lambda name: name in selected_model_name)
].iloc[0]
selected_parameters = literal_eval(parameter_row["best_parameters"])

print("Selected model:", selected_model_name)
print("Selected parameters:", selected_parameters)

Selected model: Tuned Logistic Regression
Selected parameters: {'classifier__C': 0.5, 'classifier__class_weight': 'balanced'}


In [4]:
development_df = pd.concat(
    [train_df, validation_df], ignore_index=True
)

if "SVM" in selected_model_name:
    final_model = build_svm_pipeline()
else:
    final_model = build_logistic_regression_pipeline()

final_model.set_params(**selected_parameters)
final_model.fit(development_df["text"], development_df["sentiment"])

print(f"Final model trained on {len(development_df):,} rows.")

Final model trained on 62,983 rows.


In [5]:
test_predictions = final_model.predict(test_df["text"])
macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        test_df["sentiment"],
        test_predictions,
        average="macro",
        zero_division=0,
    )
)
weighted_f1 = precision_recall_fscore_support(
    test_df["sentiment"],
    test_predictions,
    average="weighted",
    zero_division=0,
)[2]

test_metrics = pd.DataFrame([{
    "model": selected_model_name,
    "test_rows": len(test_df),
    "accuracy": accuracy_score(test_df["sentiment"], test_predictions),
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
}])

test_metrics

,model,test_rows,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,Tuned Logistic Regression,998,0.6002,0.590522,0.585965,0.583848,0.595903


In [6]:
test_report = pd.DataFrame(classification_report(
    test_df["sentiment"],
    test_predictions,
    output_dict=True,
    zero_division=0,
)).transpose()
labels = sorted(test_df["sentiment"].unique())
test_confusion = pd.DataFrame(
    confusion_matrix(test_df["sentiment"], test_predictions, labels=labels),
    index=[f"actual_{label}" for label in labels],
    columns=[f"predicted_{label}" for label in labels],
)

display(test_report)
display(test_confusion)

,precision,recall,f1-score,support
Irrelevant,0.496644,0.430233,0.461059,172.0000
Negative,0.591195,0.709434,0.644940,265.0000
Neutral,0.653333,0.515789,0.576471,285.0000
Positive,0.620915,0.688406,0.652921,276.0000
accuracy,0.600200,0.600200,0.600200,0.6002
macro avg,0.590522,0.585965,0.583848,998.0000
weighted avg,0.600864,0.600200,0.595903,998.0000


,predicted_Irrelevant,predicted_Negative,predicted_Neutral,predicted_Positive
actual_Irrelevant,74,42,17,39
actual_Negative,17,188,28,32
actual_Neutral,34,59,147,45
actual_Positive,24,29,33,190


In [7]:
test_metrics.to_csv(results_dir / "final_test_metrics.csv", index=False)
test_report.to_csv(results_dir / "final_test_classification_report.csv")
test_confusion.to_csv(results_dir / "final_test_confusion_matrix.csv")

In [8]:
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "sentiment_pipeline.joblib"

joblib.dump(final_model, model_path)
loaded_model = joblib.load(model_path)

assert (
    final_model.predict(test_df["text"].head(20))
    == loaded_model.predict(test_df["text"].head(20))
).all()

print(f"Saved and reloaded model: {model_path}")

Saved and reloaded model: C:\Users\jadka\OneDrive\Documents\progressSoft_internship\phase-1-machine-learning-nlp\assignment\models\sentiment_pipeline.joblib


In [9]:
example_texts = [
    "I absolutely love this new update!",
    "This game is terrible and completely broken.",
    "The maintenance update begins tomorrow.",
]
example_predictions = pd.DataFrame({
    "text": example_texts,
    "predicted_sentiment": loaded_model.predict(example_texts),
})
example_predictions.to_csv(
    results_dir / "example_predictions.csv", index=False
)

example_predictions

,text,predicted_sentiment
0,I absolutely love this new update!,Positive
1,This game is terrible and completely broken.,Negative
2,The maintenance update begins tomorrow.,Neutral


In [10]:
from src.api import app

api_client = TestClient(app)
health_response = api_client.get("/")
prediction_response = api_client.post(
    "/predict", json={"text": example_texts[0]}
)
blank_response = api_client.post("/predict", json={"text": "   "})

assert health_response.status_code == 200
assert prediction_response.status_code == 200
assert blank_response.status_code == 422

print(health_response.json())
print(prediction_response.json())
print(blank_response.json())

{'message': 'Twitter Sentiment API is running'}
{'text': 'I absolutely love this new update!', 'predicted_sentiment': 'Positive'}
{'detail': 'Text must not be blank'}


The final score is measured on group-disjoint test text that was not used for fitting, cross-validation, or model selection. The saved pipeline and API return the same predictions, and blank requests are rejected.